In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import signac
import cg_pyrosetta
import numpy as np
import pandas as pd
import copy
import os
import flow
import analyze_foldamers
import matplotlib.pyplot as plt
from matplotlib import cm
import mdtraj as md
from flow import FlowProject
plt.rcParams["font.family"] = "serif"
plt.rcParams.update({"font.size" : 15})

# Jupyter Notebook for developing analysis for CG PyRosetta simulations

# These analyses should be saved and eventually moved into their own dedicated functions
# Also all figures should be saved as their own PDFs for use in papers

## Load in Signac project
Here we're using the signac python API to pull in the project in this working directory. Using the `signac.detec_schema()` command we are able to pull all the state points of each simulation we ran.

In [ ]:
for item in schema.items():
    print(item[1].values())

In [ ]:
project = signac.get_project()
schema = project.detect_schema()
sc_sizes, reps = schema.items()
sc_sizes = list(sc_sizes[1][float])
sc_sizes.sort()

In [ ]:
# Show all SC sizes we ran
sc_sizes

We can now iterate over all simulations and sort them based on which state point variable they were simualating. We now have a list of all directories in the `workspace` that correspond to each type of SC size. 

In [ ]:
for sc_size in sc_sizes:
    print("SC Size = " + str(round(sc_size, 4)))
    for job in project.find_jobs({'sc_size':sc_size}):
        print(job.id)

In [ ]:
kts = np.array([10 * ( 0.9 ) ** i for i in range(50)])
out_steps = np.array([10000 * i for i in range(50)])
print(kts)

In [ ]:
for sc_size in sc_sizes:
    fig, ax1 = plt.subplots(1,1,figsize = [20,10])
    ax2 = ax1.twinx()
    ax2.plot(out_steps, kts, 'r')
    ax2.set_xlim([0, 500000])
    ax2.set_ylabel("Simulated Temperature", color = 'r')
    for job in project.find_jobs({'sc_size':sc_size}):
        energy_file = job.fn("energies.txt")
        if os.path.isfile(energy_file):
            energies = pd.read_csv(energy_file, header=None)
            ax1.plot(energies.values[:,0], energies.values[:,1], alpha = 0.4, lw=2)
            ax1.set_xlabel("Steps")
            ax1.set_ylabel("Energy (A.E.U)")
            # ax1.set_xlim([0, 500000])
            ax1.set_title("SC size = " + str(round(job.sp['sc_size'],4)))
        else:
            print("sc_size", sc_size, "hasn't started yet")    


In [ ]:
sc_sizes

In [ ]:
# DBSCAN Clustering
test = [sc_sizes[2]] #temporary
for sc_size in test:
    traj_file_list = []
    all_energies = []
    energy_traj = []
    for job in project.find_jobs({'sc_size':sc_size}):
        structure_file = job.fn('trajectory.pdb')
        traj_file_list.append(structure_file)
        energy_file = job.fn('energies.txt')
        energies = pd.read_csv(energy_file, header = None)
        energy_traj.append(energies.values[:, 1])
        all_energies.extend(energies.values[:, 1])
    all_energies = np.array(all_energies)

In [ ]:
medoid_positions, cluster_sizes, cluster_rmsd, n_noise, silhouette_avg, labels, original_indices = analyze_foldamers.cluster.\
get_cluster_medoid_positions_DBSCAN(traj_file_list, eps = 0.15, min_samples = 30, frame_start = 700, frame_stride = 1, output_dir="cluster_output", cgmodel=None, plot_silhouette=True, filter=True, filter_ratio = 0.5, output_cluster_traj = True)

In [ ]:
len(original_indices)

In [ ]:
len(labels)

In [ ]:
cluster_sizes

In [ ]:
clusters = np.unique(labels)
for i in clusters[1:]:
    plt.figure(figsize = [5, 5], dpi=100)
    cluster_indices = np.where(labels == i)[0]
    cluster_energies = all_energies[original_indices[cluster_indices]]
    print(len(cluster_energies))
    print(cluster_sizes[i])
    mean_energy = np.mean(cluster_energies)
    std_energy = np.std(cluster_energies)
    color = cm.nipy_spectral(float(i) / len(clusters[1:]))
    plt.hist(cluster_energies, bins = 50, color='red')
    plt.xlabel("Mean Energy (A.E.U)")
    plt.ylabel("Energy Std. Err. (A.E.U)")
    plt.title("Cluster " + str(i)+ " Energy Distribution")

In [ ]:
std_energy

In [ ]:
len(all_energies)

In [ ]:
clusters[1], clusters[2] = clusters[2], clusters[1]
clusters

In [ ]:
fig = plt.figure(figsize = [10, 10], dpi=500)
ax = fig.add_axes([0,0,1,1])
clusters = np.unique(labels)
energy_dists = []
for i in clusters[1:]:
    cluster_indices = np.where(labels == i)[0]
    cluster_energies = all_energies[original_indices[cluster_indices]]
    energy_dists.append(cluster_energies)

# 
# plt.errorbar(cluster_rmsd[i], avg_energy, c = color, yerr=std_energy, fmt='o', markersize=10, capsize=5)
parts = ax.violinplot(energy_dists, positions = cluster_rmsd, showextrema=False, showmeans=False, widths = 0.02)
for i, pc in enumerate(parts['bodies']):
    color = cm.nipy_spectral(float(i) / len(clusters[1:]))
    pc.set_facecolor(color)
    pc.set_edgecolor('black')
    pc.set_alpha(0.5)
# quartile1, medians, quartile3 = np.percentile(energy_dists, [25, 50, 75])
# plt.scatter(medians, marker = 'o')
plt.xlabel("Cluster RMSF (A.L.U)")
plt.ylabel("Cluster Mean Energy (A.E.U)")
plt.legend(["Cluster "+ str(a) for a in clusters[1:]], loc = "best")

In [ ]:
# Get all simulation trajectories in a list of trajectories/in a single trajectory
sim_traj = []
all_traj = None
for job in project.find_jobs({'sc_size':sc_sizes[2]}):
    job_traj = md.load(job.fn("trajectory.pdb"))
    sim_traj.append(job_traj)
    if all_traj is None:
        all_traj = job_traj
    else:
        all_traj = all_traj.join(job_traj)

In [ ]:
plt.figure(figsize = [10, 10], dpi=500)
clusters = np.unique(labels)
energy_stdevs = []
colors = ["k", "r", "b", "c"]
for i in clusters[1:]:
    medoid_mdtraj = md.load("cluster_output/medoid_" + str(i) + ".pdb")
    cluster_indices = np.where(labels == i)[0]
    rmsds_medoid_i = md.rmsd(all_traj[original_indices[cluster_indices]], medoid_mdtraj)
    cluster_energies = all_energies[original_indices[cluster_indices]]
    color = cm.nipy_spectral(float(i) / len(clusters[1:]))
    plt.scatter(rmsds_medoid_i, cluster_energies, s=1, alpha=0.7, c=color)
    plt.xlabel("RMSD to Cluster Medoid (A.L.U)")
    plt.ylabel("Cluster Energies (A.E.U)")
plt.legend(["Cluster "+ str(a) for a in clusters[1:]], loc = "best")

In [ ]:
energy_stdevs

In [ ]:
cluster_rmsd

In [ ]:
# Distribution of minimum energy structures
sorted_energies = copy.deepcopy(all_energies)
sorted_energies.sort()
plt.hist(sorted_energies[:5000], bins=30)

In [ ]:
all_traj = md.load(traj_file_list)

In [ ]:
cluster_indices

In [ ]:
traj = md.load(job.fn("trajectory.pdb"))
traj = traj.join(md.load(job.fn("trajectory.pdb")))
traj

In [ ]:
for i in clusters[1:]:
    plt.figure(figsize = [10,10])
    for j in range(len(sim_traj)):
        cluster_indices = np.where(labels == i)[0]
        medoid_mdtraj = md.load("cluster_output/medoid_" + str(i) + ".pdb")
        rmsds_medoid_i = md.rmsd(sim_traj[j], medoid_mdtraj)
        plt.scatter(rmsds_medoid_i, energy_traj[j], s = 10, alpha = 0.5)
        plt.xlabel("RMSD to Mediod " + str(i+1))
        plt.ylabel("Energies")
        plt.title("RMSD to Medoid " + str(i))
    plt.savefig(job.fn("kmedoid_rmsd_energy_scatter.jpg"), dpi=600)
    

In [ ]:
# RMSD (to minimum energy structure) vs Energy

for sc_size in sc_sizes:
    plt.figure(figsize = [20,10])
    all_traj = md.load(traj_file_list)

    for job in project.find_jobs({'sc_size':sc_size}):
        # Get MC traj
        structure_file = job.fn("structure.txt")
        struct_loc = pd.read_csv(structure_file, header=None)
        traj = md.load([job.fn(file) for file in struct_loc[1].values])

        # Get minimum energy structure
        energy_file = job.fn("energies.txt")
        energies = pd.read_csv(energy_file, header=None).values[:,1]
        
        min_energy_index = np.where(all_energies == np.min(all_energies))
        min_energy_struct = all_traj.slice(min_energy_index)

        # Compute RMSDs to trajectory
        rmsds_medoid_i = md.rmsd(traj, min_energy_struct)

        plt.scatter(rmsds_medoid_i, energies, s = 1)
        
        
        
    # Get ALL MC traj
    structure_file = job.fn("structure.txt")
    struct_loc = pd.read_csv(structure_file, header=None)
    traj = md.load(traj_file_list)

    # Get minimum energy structure
    min_energy_index= np.where(all_energies == np.min(all_energies))
    min_energy_struct = traj.slice(min_energy_index)

    # Compute RMSDs to trajectory
    rmsds_medoid_i = md.rmsd(traj, min_energy_struct)
    
    # plt.figure(figsize = [20,10])
    # plt.scatter(rmsds_medoid_i, all_energies, s = 1)

        
    # Minimum energy structure
    
    minimum_index = min_energy_index[0][0]
    poi_1 = np.where(all_energies == sorted_energies[450])[0][0]
    poi_2 = np.where(all_energies == sorted_energies[471])[0][0]
    poi_3 = np.where(all_energies == sorted_energies[1400])[0][0]
    poi_4 = np.where(all_energies == sorted_energies[2300])[0][0]
    poi_5 = np.where(all_energies == sorted_energies[700])[0][0]
    
    min_energy_struct.save_pdb("overall_minimim.pdb")
    plt.scatter(rmsds_medoid_i[minimum_index], all_energies[minimum_index], c = 'r', s = 150, marker = 'D')
    plt.scatter(rmsds_medoid_i[poi_1], all_energies[poi_1], c = 'g', s = 150, marker = 'D')
    plt.scatter(rmsds_medoid_i[poi_2], all_energies[poi_2], c = 'y', s = 150, marker = 'D')
    plt.scatter(rmsds_medoid_i[poi_3], all_energies[poi_3], c = 'k', s = 150, marker = 'D')
    plt.scatter(rmsds_medoid_i[poi_4], all_energies[poi_4], c = 'm', s = 150, marker = 'D')
    plt.scatter(rmsds_medoid_i[poi_5], all_energies[poi_5], c = 'C1', s = 150, marker = 'D')
    legend_list = ["rep " + str(i) for i in range(10)]
    legend_list.extend(["Structure A", "Structure B", "Structure C", "Structure D", "Structure E", "Structure F"])
    plt.legend(legend_list)
    plt.xlabel("RMSD to Minimum Energy Structure")
    plt.ylabel("Energies")
    plt.savefig("min_energy_rmsd_energy_scatter.jpg", dpi=600)



In [ ]:
# Minimum energy structure

minimum_index = min_energy_index[0][0]
poi_1 = np.where(all_energies == sorted_energies[450])[0][0]
poi_2 = np.where(all_energies == sorted_energies[471])[0][0]
poi_3 = np.where(all_energies == sorted_energies[1400])[0][0]
poi_4 = np.where(all_energies == sorted_energies[2300])[0][0]
poi_5 = np.where(all_energies == sorted_energies[700])[0][0]

print("Minimum energy structure location:", traj_file_list[minimum_index])
print("Point of Interest 1 location:", traj_file_list[poi_1])
print("Point of Interest 2 location:", traj_file_list[poi_2])
print("Point of Interest 3 location:", traj_file_list[poi_3])
print("Point of Interest 4 location:", traj_file_list[poi_4])
print("Point of Interest 5 location:", traj_file_list[poi_5])



min_energy_struct.save_pdb("overall_minimim.pdb")
plt.figure(figsize = [20,10])
plt.scatter(rmsds_medoid_i, all_energies, s = 10, label = None)
plt.scatter(rmsds_medoid_i[minimum_index], all_energies[minimum_index], c = 'r', s = 150, marker = 'D')
plt.scatter(rmsds_medoid_i[poi_1], all_energies[poi_1], c = 'g', s = 150, marker = 'D')
plt.scatter(rmsds_medoid_i[poi_2], all_energies[poi_2], c = 'y', s = 150, marker = 'D')
plt.scatter(rmsds_medoid_i[poi_3], all_energies[poi_3], c = 'k', s = 150, marker = 'D')
plt.scatter(rmsds_medoid_i[poi_4], all_energies[poi_4], c = 'm', s = 150, marker = 'D')
plt.scatter(rmsds_medoid_i[poi_5], all_energies[poi_5], c = 'C1', s = 150, marker = 'D')
plt.legend(["All structures", "Structure A", "Structure B", "Structure C", "Structure D", "Structure E", "Structure F"])
plt.xlabel("RMSD to Minimum Energy Structure")
plt.ylabel("Energy (A.E.U)")
plt.savefig("min_energy_rmsd_energy_scatter.jpg", dpi=600)

In [ ]:
rmsd_matrix = np.zeros([traj.n_frames, traj.n_frames])
for i in range(traj.n_frames):
    rmsd_matrix[:, i] = md.rmsd(traj, traj.slice(i))

In [ ]:
plt.hist(rmsd_matrix.flatten(), bins = 100)
plt.ylabel("Count")
plt.xlabel("RMSDs")